# Cross Encoder Reranking

In [ ]:
import os
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sentence_transformers import CrossEncoder

# Root path setup for imports
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[0]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Loading retrieval and score computation function
from src.retrieval import retrieve_top_k, rerank

In [2]:
# Configuration
data_path = PROJECT_ROOT / "data/processed/arxiv_papers.csv"
embeddings_path = PROJECT_ROOT / "data/processed/sbert_embeddings.npy"

data = pd.read_csv(data_path)
sbert_embeddings = np.load(embeddings_path)
print(f"Dimensions of SBERT embeggings: ", sbert_embeddings.shape)

Dimensions of SBERT embeggings:  (1000, 384)


## SBERT + Cross Encoder

In [ ]:
# Load Cross encoder
cross_encoder_stsb = CrossEncoder("cross-encoder/stsb-roberta-base")
cross_encoder_msmarco = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Load SBERT model
sbert = SentenceTransformer("all-MiniLM-L6-v2")

In [10]:
# Example query for retrieval
query = "self-supervised learning for image representation"

# Bi-encoder candidate Retrieval
candidates = retrieve_top_k(query=query, preferred_category=None, df=data, embeddings=sbert_embeddings, model=sbert, k=20)

# 2 Stage retrieval for SBERT
reranked_sbert_stsb = rerank(query, candidates, cross_encoder_stsb)
reranked_sbert_msmarco = rerank(query, candidates, cross_encoder_msmarco)

reranked_sbert_stsb = reranked_sbert_stsb.sort_values("cross_score", ascending=False)
reranked_sbert_msmarco = reranked_sbert_msmarco.sort_values("cross_score", ascending=False)

In [11]:
# Reranked results
display(reranked_sbert_stsb[['title','summary','bi_score','cross_score']].head())
display(reranked_sbert_msmarco[['title','summary','bi_score','cross_score']].head())

,title,summary,bi_score,cross_score
2,Learning Representations by Maximizing Mutual ...,We propose an approach to self-supervised repr...,0.559654,0.718593
5,Self-supervised Body Image Acquisition Using a...,This work investigates how a naive agent can a...,0.464249,0.699335
8,Uncovering the structure of clinical EEG signa...,Objective. Supervised learning paradigms are o...,0.442418,0.674343
9,An empirical study of domain-agnostic semi-sup...,A class of recent semi-supervised learning (SS...,0.435032,0.668178
1,A Framework For Contrastive Self-Supervised Le...,Contrastive self-supervised learning (CSL) is ...,0.562472,0.663573


,title,summary,bi_score,cross_score
2,Learning Representations by Maximizing Mutual ...,We propose an approach to self-supervised repr...,0.559654,8.193757
0,CompRess: Self-Supervised Learning by Compress...,Self-supervised learning aims to learn good re...,0.623800,8.089027
1,A Framework For Contrastive Self-Supervised Le...,Contrastive self-supervised learning (CSL) is ...,0.562472,7.607416
6,Look Listen and Attend: Co-Attention Network f...,When watching videos the occurrence of a visua...,0.460468,6.549674
3,Functional Regularization for Representation L...,Unsupervised and self-supervised learning appr...,0.522539,5.937018


MS-MARCO cross-encoder sgnificantly improves ranking by modeling query-document relevance, making it more suitable for semantic search

## SPECTER + Cross Encoder

### SPECTER is trained on scientific paper semantics

In [ ]:
# Load specter model
specter = SentenceTransformer('allenai-specter')

specter_embeddings_path = PROJECT_ROOT / 'data/processed/specter_embeddings.npy'

# Generate/load embeddings
if not os.path.exists(specter_embeddings_path):
    specter_embeddings = specter.encode(data['combined'].tolist(), show_progress_bar=True)
    np.save(specter_embeddings_path, specter_embeddings)
else:
    specter_embeddings = np.load(specter_embeddings_path)

print(f"Dimensions of SPECTER embeddings: ", specter_embeddings.shape)

In [15]:
# 2 Stage retrieval for SPECTER + MS-MARCO
candidates_specter = retrieve_top_k(query=query, preferred_category=None, df=data, embeddings=specter_embeddings, model=specter, k=20)
reranked_specter = rerank(query, candidates_specter, cross_encoder_msmarco).sort_values("cross_score", ascending=False)
reranked_specter[['title','summary','bi_score','cross_score']].head()

,title,summary,bi_score,cross_score
7,Learning Representations by Maximizing Mutual ...,We propose an approach to self-supervised repr...,0.783408,8.193757
18,A Framework For Contrastive Self-Supervised Le...,Contrastive self-supervised learning (CSL) is ...,0.772665,7.607416
16,Deep Convolutional Transform Learning -- Exten...,This work introduces a new unsupervised repres...,0.773920,-2.068776
0,A generalized linear joint trained framework f...,The elastic-net is among the most widely used ...,0.819922,-4.420799
17,Optimization for Supervised Machine Learning: ...,Many key problems in machine learning and data...,0.773639,-5.591026


In [17]:
# Comparison
comparison = pd.DataFrame({
    'SBERT Top-5': reranked_sbert_msmarco['title'].head(5).values,
    'SPECTER Top-5': reranked_specter['title'].head(5).values
})

with pd.option_context('display.max_colwidth', None):
    display(comparison.head())

,SBERT Top-5,SPECTER Top-5
0,Learning Representations by Maximizing Mutual Information Across Views,Learning Representations by Maximizing Mutual Information Across Views
1,CompRess: Self-Supervised Learning by Compressing Representations,A Framework For Contrastive Self-Supervised Learning And Designing A New Approach
2,A Framework For Contrastive Self-Supervised Learning And Designing A New Approach,Deep Convolutional Transform Learning -- Extended version
3,Look Listen and Attend: Co-Attention Network for Self-Supervised Audio-Visual Representation Learning,A generalized linear joint trained framework for semi-supervised learning of sparse features
4,Functional Regularization for Representation Learning: A Unified Theoretical Perspective,Optimization for Supervised Machine Learning: Randomized Algorithms for Data and Parameters


## Comparison between different approachs
- **TFIDF - Cosine Similarity**
- **SBERT + Crosss Encoder**
- **SPECTER + Crosss Encoder**

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

tfidf = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = tfidf.fit_transform(data['combined'])

query_vec = tfidf.transform([query])
data['tfidf_score'] = cos_sim(query_vec, tfidf_matrix)[0]
tfidf_ranked = data.sort_values('tfidf_score', ascending=False).reset_index(drop=True)

# ranks
def get_rank(df, target, title_col='title'):
    match = df[df[title_col].str.contains(target, case=False)]
    return match.index[0] + 1 if len(match) > 0 else "Not in results"

In [22]:
queries = [
    "learning visual features without labels",
    "preventing overfitting in deep neural networks",
    "speeding up transformer model training",
    "handling bias in text classification models",
    "making neural networks faster and smaller"
]

# Target papers — manually identify one ground truth per query
targets = {
    "learning visual features without labels": "Learning Representations by Maximizing Mutual Information",
    "preventing overfitting in deep neural networks": "MaxDropout: Deep Neural Network Regularization Based on Maximum Output Values",
    "speeding up transformer model training": "Accelerating Training of Transformer-Based Language Models",
    "handling bias in text classification models": "Robustness to Spurious Correlations in Text Classification ",
    "making neural networks faster and smaller": "Early Stopping in Deep Networks: Double Descent and How to Eliminate it"
}

results_table = []

for query in queries:

    # Reranking on SBERT embeddings
    candidates_sbert = retrieve_top_k(query=query, preferred_category=None, df=data, embeddings=sbert_embeddings, model=sbert, k=20)
    reranked_sbert = rerank(query, candidates_sbert, cross_encoder_msmarco).sort_values("cross_score", ascending=False)

    # Reranking on SPECTER embeddings
    candidates_specter = retrieve_top_k(query=query, preferred_category=None, df=data, embeddings=specter_embeddings, model=specter, k=20)
    reranked_specter = rerank(query, candidates_specter, cross_encoder_msmarco).sort_values("cross_score", ascending=False)

    tfidf_rank   = get_rank(tfidf_ranked, targets[query])
    sbert_rank   = get_rank(reranked_sbert, targets[query])
    specter_rank = get_rank(reranked_specter, targets[query])
    
    results_table.append({
        "Query": query,
        "TF-IDF Rank": tfidf_rank,
        "SBERT+CE Rank": sbert_rank,
        "SPECTER+CE Rank": specter_rank
    })

pd.DataFrame(results_table)

,Query,TF-IDF Rank,SBERT+CE Rank,SPECTER+CE Rank
0,learning visual features without labels,9,18,2
1,preventing overfitting in deep neural networks,106,4,7
2,speeding up transformer model training,974,1,Not in results
3,handling bias in text classification models,1000,1,1
4,making neural networks faster and smaller,963,10,5


#### Note:
- **TF-IDF**: Plain cosine similarity on TF-IDF vector matrix across full text.
- **SBERT**: Title and abstract are encoded separately, combined as weighted embeddings (0.6 title + 0.4 abstract), then cosine similarity is applied on the resulting vector — giving more weight to the title which carries the core idea.
- **SPECTER**: Title and abstract are concatenated as combined text before encoding, consistent with SPECTER's original pretraining input format, then cosine similarity is applied.